**_3. WAP to implement a three-layer neural network using Tensor flow library (only, no keras) to classify MNIST handwritten digits dataset. Demonstrate the implementation of feed-forward and back-propagation approaches._**

In [11]:
import tensorflow as tf
import tensorflow_datasets as tfds

# Load MNIST dataset
mnist, info = tfds.load('mnist', with_info=True, as_supervised=True)
mnist_train, mnist_test = mnist['train'], mnist['test']

# Define network parameters
n_input = 784  # 28x28 pixels
n_hidden1 = 128  # First hidden layer
n_hidden2 = 64   # Second hidden layer
n_output = 10    # Number of classes (digits 0-9)
learning_rate = 0.01
epochs = 10
batch_size = 100

# Preprocess the dataset
def preprocess_image(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.reshape(image, [-1, 784])
    label = tf.one_hot(label, n_output)
    return image, label

train_dataset = mnist_train.map(preprocess_image).shuffle(60000).batch(batch_size)
test_dataset = mnist_test.map(preprocess_image).batch(batch_size)

# Initialize weights and biases
class NeuralNetwork(tf.Module):
    def __init__(self):
        self.weights = {
            'h1': tf.Variable(tf.random.normal([n_input, n_hidden1])),
            'h2': tf.Variable(tf.random.normal([n_hidden1, n_hidden2])),
            'out': tf.Variable(tf.random.normal([n_hidden2, n_output]))
        }
        self.biases = {
            'b1': tf.Variable(tf.random.normal([n_hidden1])),
            'b2': tf.Variable(tf.random.normal([n_hidden2])),
            'out': tf.Variable(tf.random.normal([n_output]))
        }

    def __call__(self, x):
        layer1 = tf.nn.relu(tf.add(tf.matmul(x, self.weights['h1']), self.biases['b1']))
        layer2 = tf.nn.relu(tf.add(tf.matmul(layer1, self.weights['h2']), self.biases['b2']))
        output_layer = tf.add(tf.matmul(layer2, self.weights['out']), self.biases['out'])
        return output_layer

# Define the model
model = NeuralNetwork()

# Define loss function (cross-entropy) and optimizer
loss_object = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

# Define accuracy
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.CategoricalAccuracy(name='train_accuracy')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.CategoricalAccuracy(name='test_accuracy')

@tf.function
def train_step(images, labels):
    with tf.GradientTape() as tape:
        predictions = model(images)
        loss = loss_object(labels, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    train_loss(loss)
    train_accuracy(labels, predictions)

@tf.function
def test_step(images, labels):
    predictions = model(images)
    t_loss = loss_object(labels, predictions)

    test_loss(t_loss)
    test_accuracy(labels, predictions)

# Train the model
# Train the model
for epoch in range(epochs):
    # Reset the metrics at the start of the next epoch
    train_loss.reset_state()
    train_accuracy.reset_state()
    test_loss.reset_state()
    test_accuracy.reset_state()

    for images, labels in train_dataset:
        train_step(images, labels)

    for test_images, test_labels in test_dataset:
        test_step(test_images, test_labels)

    print(
        f"Epoch {epoch+1}, "
        f"Loss: {train_loss.result():.4f}, "
        f"Accuracy: {train_accuracy.result():.4f}, "
        f"Test Loss: {test_loss.result():.4f}, "
        f"Test Accuracy: {test_accuracy.result():.4f}"
    )

print("Training complete!")



RuntimeError: `tf.data.Dataset` only supports Python-style iteration in eager mode or within tf.function.